# Compiling Amended Regulations — reference solution

An agency publishes an instruction in prose — *"Amend § X by revising paragraph
(a)(3)"* — and somebody has to carry it out against the regulation as it stands.
This notebook does that mechanically.

There is nothing to train. The whole problem is resolving a natural-language
address against a hierarchical document, so the work is in three places:

1. cutting the section into its designated paragraphs,
2. telling a paragraph opening from a cross-reference to one, and
3. handling `* * *`, which means "the rest of this paragraph is unchanged".

Each stage below is measured on held-out CFR parts before anything is submitted.
CPU only, a couple of minutes end to end.


In [ ]:
import json, re, difflib, random
from pathlib import Path
import pandas as pd

DATA = next((p for p in (Path("/data"), Path("dataset/public"),
                         Path("/kaggle/input/recompile-dataset/public"))
             if (p / "test.csv").exists()), Path("dataset/public"))

train = pd.read_csv(DATA / "train.csv", dtype=str, keep_default_na=False)
test = pd.read_csv(DATA / "test.csv", dtype=str, keep_default_na=False)

def load_item(item_id):
    with open(DATA / "items" / f"{item_id}.json", encoding="utf8") as fh:
        return json.load(fh)

items = {i: load_item(i) for i in list(train["item_id"]) + list(test["item_id"])}
gold = dict(zip(train["item_id"], train["after_text"]))
print(f"{len(train)} labelled items, {len(test)} test items")
print(f"median section length: "
      f"{sorted(len(items[i]['before_text'].split()) for i in items)[len(items)//2]} tokens")

## The metric, so we can measure as we go

Comparing our text to the official text directly would tell us nothing: an
amendment moves a few dozen words inside a section of several hundred, so doing
nothing at all scores about 0.97 on any similarity measure. We compare **edit
scripts** instead, weighting each operation by the number of tokens it moves.

In [ ]:
UNI = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
       "\u2013": "-", "\u2014": "-", "\u2212": "-", "\u00a0": " ",
       "\u2026": "...", "\u00b6": " "}

def normalise(text):
    text = "" if text is None else str(text)
    for k, v in UNI.items():
        text = text.replace(k, v)
    return " ".join(text.split())

def edit_ops(before, after):
    a, b = normalise(before).split(), normalise(after).split()
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    return {(i1, i2, " ".join(b[j1:j2]))
            for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"}

def weight(op):
    i1, i2, inserted = op
    return (i2 - i1) + len(inserted.split())

def score(before, gold_after, pred_after):
    g, p = edit_ops(before, gold_after), edit_ops(before, pred_after)
    if not g and not p:
        return 1.0
    matched = sum(weight(o) for o in g & p)
    if not matched:
        return 0.0
    prec = matched / sum(weight(o) for o in p)
    rec = matched / sum(weight(o) for o in g)
    return 2 * prec * rec / (prec + rec)

def evaluate(ids, fn):
    total = 0.0
    for i in ids:
        it = items[i]
        try:
            pred = fn(it["before_text"], it["blocks"]) or it["before_text"]
        except Exception:
            pred = it["before_text"]
        total += score(it["before_text"], gold[i], pred)
    return total / len(ids)

## A held-out split

The labelled items are our only measuring stick, and the split has to respect the
same boundary the real one does. The real split is disjoint by CFR part, because
agencies re-run the same amendment shape year after year and an item-level split
would put one year's version on each side. Each item carries a `group_id` naming
its part, so we hold out whole groups.

In [ ]:
rng = random.Random(11)
groups = sorted({items[i]["group_id"] for i in train["item_id"]})
rng.shuffle(groups)
val_groups = set(groups[int(len(groups) * 0.75):])

fit_ids = [i for i in train["item_id"] if items[i]["group_id"] not in val_groups]
val_ids = [i for i in train["item_id"] if items[i]["group_id"] in val_groups]
assert not ({items[i]["group_id"] for i in fit_ids}
            & {items[i]["group_id"] for i in val_ids})
print(f"{len(fit_ids)} items for inspection, {len(val_ids)} held out "
      f"across {len(val_groups)} disjoint CFR parts")

## Stage 1 — splice the printed block over the addressed paragraph

The obvious first attempt. Find "paragraph (a)(3)" in the instruction, find `(a)(3)`
in the section, replace everything from there to the next designator with what the
agency printed.

In [ ]:
SIMPLE_ADDR = re.compile(r"paragraphs?\s+((?:\([0-9A-Za-z]{1,4}\))+)", re.I)
ANY_DESIG = re.compile(r"^\((?:[0-9A-Za-z]{1,4})\)")

def naive_apply(before_text, blocks):
    toks = before_text.split()
    for blk in blocks:
        printed = (blk.get("printed") or "").strip()
        if not printed:
            continue
        for instr in blk["instructions"]:
            m = SIMPLE_ADDR.search(instr)
            if not m:
                continue
            target = m.group(1)
            for i, t in enumerate(toks):
                if t.startswith(target):
                    end = len(toks)
                    for j in range(i + 1, len(toks)):
                        if ANY_DESIG.match(toks[j]):
                            end = j
                            break
                    toks = toks[:i] + printed.split() + toks[end:]
                    break
    return " ".join(toks)

print(f"stage 1, held out: {evaluate(val_ids, naive_apply):.4f}")

That is weak, and the two reasons are worth seeing directly.

**Cross-references.** Regulations cite their own paragraphs constantly — *"except
as provided in paragraph (d) of this section"* — and those look exactly like
paragraph openings. Stopping at "the next designator of any kind" therefore stops
in the middle of a sentence.

**Levels.** A paragraph ends at the next designator *at the same or a shallower
level*, so `(a)(3)` does not end at `(i)` nested inside it. And whether `(i)` is a
first-level letter after `(h)` or a third-level roman numeral cannot be read off
the character — only off what preceded it.

In [ ]:
probe = items[fit_ids[0]]["before_text"].split()
hits = [(i, t, probe[i-1] if i else "<start>")
        for i, t in enumerate(probe) if ANY_DESIG.match(t)]
for i, t, prev in hits[:8]:
    looks_like_opening = prev[-1:] in ".:;?!" or i == 0
    print(f"  {t:10s} after {prev!r:24s} -> "
          f"{'paragraph opening' if looks_like_opening else 'cross-reference'}")

## Stage 2 — a real paragraph hierarchy

Assign every designator a level by the sequence it continues, and treat a
designator as an opening only when the preceding token ends a sentence.

In [ ]:
"""Reference solution: apply Federal Register amendatory instructions to a CFR
section.

The shape of the problem, in the order the code deals with it:

  1. Cut the section into its designated paragraphs -- (a), (a)(1), (a)(1)(i) ...
     CFR designators cycle lower-alpha / digit / lower-roman / upper-alpha, and
     "(i)" is a lower-alpha designator following "(h)" about as often as it is a
     roman numeral opening a third level, so the level has to be inferred from
     what came before rather than from the character.
  2. Read the address out of each instruction.
  3. Match each address to the block of replacement text the agency printed under
     it -- which is keyed by the same designators, and which may be partial:
     "* * *" means "the rest of this paragraph is unchanged".
  4. Apply, in order.
"""
import re

DESIG_RUN = re.compile(r"^((?:\([0-9A-Za-z]{1,4}\))+)")
DESIG_ONE = re.compile(r"\(([0-9A-Za-z]{1,4})\)")
ROMAN = re.compile(r"^[ivxlcdm]+$")
ELLIPSIS = "* * *"


def kind_of(d):
    """CFR designators cycle through these levels."""
    if d.isdigit():
        return "num"
    if d.isupper():
        return "ROMAN" if ROMAN.match(d.lower()) and len(d) > 1 else "ALPHA"
    if ROMAN.match(d) and d not in ("i", "v", "x", "l", "c", "d", "m"):
        return "roman"
    return "alpha"          # single ambiguous letters resolve by context below


LADDER = ["alpha", "num", "roman", "ALPHA", "ROMAN"]


ROMAN_ORDER = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x",
               "xi", "xii", "xiii", "xiv", "xv", "xvi", "xvii", "xviii", "xix",
               "xx", "xxi", "xxii", "xxiii", "xxiv", "xxv"]


def _roman_next(prev, cur):
    p, c = prev.lower(), cur.lower()
    return (p in ROMAN_ORDER and c in ROMAN_ORDER
            and ROMAN_ORDER.index(c) == ROMAN_ORDER.index(p) + 1)


def _next_in_seq(prev, cur, kind):
    """Is `cur` the designator that follows `prev` at this level?

    The roman fallback is load-bearing.  A lone "(i)" is indistinguishable from
    the letter i, so it gets recorded as an alpha level; when "(ii)" arrives the
    alpha test fails on length and, without this fallback, "(ii)" opens a bogus
    level deeper -- which is precisely where addresses like "paragraph (a)(2)(ii)"
    then fail to resolve.  Checking roman succession regardless of the recorded
    kind costs nothing: "(h)" -> "(i)" still matches as alpha first, and no pair
    satisfies both tests.
    """
    if kind == "num":
        if prev.isdigit() and cur.isdigit() and int(cur) == int(prev) + 1:
            return True
    elif kind in ("alpha", "ALPHA"):
        if len(prev) == 1 and len(cur) == 1 and \
                ord(cur.lower()) == ord(prev.lower()) + 1:
            return True
    return _roman_next(prev, cur)


# Words that end in a full stop without ending a sentence.  Without these,
# "see Pub. L. 95-521 (b)" reads as paragraph (b) opening.
ABBREV_LEAD = {"no", "nos", "sec", "secs", "pub", "l", "u.s.c", "u.s", "cf",
               "e.g", "i.e", "fig", "app", "pt", "ch", "art", "para", "subpt",
               "st", "stat", "jan", "feb", "mar", "apr", "jun", "jul", "aug",
               "sep", "sept", "oct", "nov", "dec", "vs", "inc", "co", "corp"}


def starts_paragraph(tokens, i):
    """Is the designator at token `i` opening a paragraph, or citing one?

    Regulations refer to their own paragraphs constantly -- "as described in
    paragraph (d) of this section" -- and those references look exactly like
    paragraph openings.  What separates them is that a real opening follows the
    *end* of the previous paragraph.  So the test is on the preceding token's
    punctuation, not on the preceding word: "of this part." ends a sentence and
    does open the next paragraph, even though "part" is a cross-reference word.
    """
    if i == 0:
        return True
    prev = tokens[i - 1]
    if prev == "*":                     # the tail of a "* * *" continuation marker
        return True
    if prev.rstrip(".,;:").lower() in ABBREV_LEAD:
        return False
    return prev.rstrip('")\'')[-1:] in ".:;?!"


def paragraph_spans(tokens):
    """{'a': (start, end), 'a.3': (start, end), ...} over token indices.

    A paragraph runs from its designator to the start of the next paragraph at
    the same or a shallower level.
    """
    marks = []                                        # (token_index, path)
    path = []                                         # list of (kind, value)
    for i, tok in enumerate(tokens):
        m = DESIG_RUN.match(tok)
        if not m or not starts_paragraph(tokens, i):
            continue
        run = DESIG_ONE.findall(m.group(1))
        first = run[0]
        k = kind_of(first)
        # Resolve the level: the deepest existing level this designator continues.
        placed = None
        for lvl in range(len(path) - 1, -1, -1):
            pk, pv = path[lvl]
            if _next_in_seq(pv, first, pk):
                placed = lvl
                k = pk
                break
        if placed is None:
            # Opening a level: "(a)" or "(1)" or "(i)" as the first of its kind.
            if first in ("a", "1", "i", "A", "I"):
                placed = len(path)
            else:
                # Out of sequence -- treat as a sibling of the deepest same-kind
                # level if there is one, otherwise as a new deepest level.
                same = [l for l in range(len(path)) if path[l][0] == k]
                placed = same[-1] if same else len(path)
        path = path[:placed] + [(k, first)]
        for extra in run[1:]:
            path.append((kind_of(extra), extra))
        marks.append((i, ".".join(v for _, v in path), len(path)))

    spans = {}
    for idx, (i, key, depth) in enumerate(marks):
        end = len(tokens)
        for j, dkey, ddepth in marks[idx + 1:]:
            if ddepth <= depth:
                end = j
                break
        else:
            end = len(tokens)
        spans.setdefault(key, (i, end))
    return spans


# --------------------------------------------------------------- the printed text

def split_printed(text):
    """Break the agency's printed block into {designator path -> text}."""
    toks = text.split()
    spans = paragraph_spans(toks)
    out = {}
    for key, (s, e) in spans.items():
        out[key] = " ".join(toks[s:e])
    if not out and text.strip():
        out[""] = text.strip()
    return out


def strip_designator(chunk):
    m = DESIG_RUN.match(chunk)
    return chunk[m.end():].strip() if m else chunk


SENT_END = re.compile(r"[.!?]$")
ABBREV = {"No.", "Nos.", "U.S.", "U.S.C.", "Sec.", "Secs.", "Pub.", "L.", "St.",
          "Inc.", "Co.", "Corp.", "Jan.", "Feb.", "Mar.", "Apr.", "Jun.", "Jul.",
          "Aug.", "Sep.", "Sept.", "Oct.", "Nov.", "Dec.", "e.g.", "i.e.", "etc.",
          "Dr.", "Mr.", "Mrs.", "Ms.", "vs.", "cf.", "Fig.", "App.", "Subpt.",
          "Pt.", "Ch.", "Div.", "Art.", "para.", "pp.", "ft.", "in.", "sq.", "min.",
          "max.", "approx.", "Stat."}


def sentences(tokens):
    """Sentence boundaries over tokens. Legal prose is thick with abbreviations
    and citations, so a naive split on '.' cuts in the wrong places."""
    out, cur = [], []
    for i, t in enumerate(tokens):
        cur.append(t)
        if not SENT_END.search(t) or t in ABBREV:
            continue
        if len(t) <= 2 and t.endswith("."):
            continue                                   # initials, "(a." fragments
        nxt = tokens[i + 1] if i + 1 < len(tokens) else None
        if nxt is not None and not (nxt[:1].isupper() or nxt.startswith("(")
                                    or nxt[:1].isdigit() or nxt.startswith('"')):
            continue
        out.append(cur)
        cur = []
    if cur:
        out.append(cur)
    return out


def merge_partial(old_tokens, new_text):
    """Splice a printed block that carries '* * *' into the existing paragraph.

    A trailing '* * *' means the agency printed only the front of the paragraph
    and the tail survives; a leading one means the opposite.
    """
    lead = new_text.strip().startswith(ELLIPSIS)
    trail = new_text.strip().endswith(ELLIPSIS)
    body = new_text.strip()
    if lead:
        body = body[len(ELLIPSIS):].strip()
    if trail:
        body = body[:-len(ELLIPSIS)].strip()
    if not lead and not trail:
        return body.split()

    old_sents = sentences(old_tokens)
    new_sents = sentences(body.split())
    if trail and not lead:
        keep = old_sents[len(new_sents):]
        return body.split() + [t for s in keep for t in s]
    if lead and not trail:
        keep = old_sents[:max(0, len(old_sents) - len(new_sents))]
        return [t for s in keep for t in s] + body.split()
    # '* * *' on both sides: the printed text replaces the middle.
    head = old_sents[:1]
    return ([t for s in head for t in s] + body.split())


# ------------------------------------------------------------------- instructions

ADDR = re.compile(
    r"paragraphs?\s+((?:\([0-9A-Za-z]{1,4}\))+(?:\s*(?:,|and|through|to)\s*"
    r"(?:\([0-9A-Za-z]{1,4}\))+)*)", re.I)
QUOTED = re.compile(r'"([^"]{1,300})"')
REMOVE_ADD = re.compile(
    r'remov\w*\s+(?:the\s+)?(?:words?|phrase|term|reference\s+to)?\s*"([^"]{1,300})"'
    r'[^"]{0,80}?(?:add\w*|insert\w*)[^"]{0,40}"([^"]{1,300})"', re.I)
REMOVE_ONLY = re.compile(
    r'remov\w*\s+(?:the\s+)?(?:words?|phrase|term)?\s*"([^"]{1,300})"', re.I)
REDESIG = re.compile(
    r"redesignat\w*\s+paragraphs?\s+((?:\([0-9A-Za-z]{1,4}\))+)"
    r"(?:\s*(?:through|to)\s*((?:\([0-9A-Za-z]{1,4}\))+))?\s*as\s+"
    r"((?:\([0-9A-Za-z]{1,4}\))+)", re.I)


def _path(desig_run):
    return ".".join(DESIG_ONE.findall(desig_run))


def parse_addresses(text):
    out = []
    for m in ADDR.finditer(text):
        for run in re.findall(r"(?:\([0-9A-Za-z]{1,4}\))+", m.group(1)):
            out.append(_path(run))
    return out


def apply_instructions(before_text, blocks):
    """Return the amended section text."""
    tokens = before_text.split()

    for blk in blocks:
        instrs = blk["instructions"]
        printed = split_printed(blk.get("printed") or "")
        for instr in instrs:
            low = instr.lower()

            m = REDESIG.search(instr)
            if m:
                src, dst = _path(m.group(1)), _path(m.group(3))
                spans = paragraph_spans(tokens)
                if src in spans:
                    s, _ = spans[src]
                    old = DESIG_RUN.match(tokens[s])
                    if old:
                        tokens[s] = ("(" + ")(".join(dst.split(".")) + ")"
                                     + tokens[s][old.end():])
                continue

            m = REMOVE_ADD.search(instr)
            if m:
                tokens = _replace_words(tokens, m.group(1), m.group(2))
                continue
            if re.search(r"\bremov\w*\b", low) and not re.search(r"\badd|insert", low):
                m = REMOVE_ONLY.search(instr)
                if m:
                    tokens = _replace_words(tokens, m.group(1), "")
                    continue
                for addr in parse_addresses(instr):
                    spans = paragraph_spans(tokens)
                    if addr in spans:
                        s, e = spans[addr]
                        tokens = tokens[:s] + tokens[e:]
                continue

            addrs = parse_addresses(instr)
            if not addrs and printed:
                addrs = [k for k in printed if k]
            for addr in addrs:
                chunk = printed.get(addr)
                if chunk is None:
                    continue
                spans = paragraph_spans(tokens)
                if addr in spans:
                    s, e = spans[addr]
                    tokens = tokens[:s] + merge_partial(tokens[s:e], chunk) + tokens[e:]
                elif re.search(r"\badd\w*\b", low):
                    tokens = _insert_paragraph(tokens, addr, chunk)
    return " ".join(tokens)


def _replace_words(tokens, old, new):
    """Quoted string substitution, returning tokens.

    Returning the joined string here instead of a token list silently turned
    `tokens` into a str for every later instruction in the same section, which
    made removals score zero and corrupted anything applied after them.
    """
    text = " ".join(tokens)
    if old in text:
        return text.replace(old, new).split()
    return tokens


def _insert_paragraph(tokens, addr, chunk):
    """Put a newly added paragraph where it belongs in the running order.

    Best evidence is the last existing sibling: "(c)(3)" goes after "(c)(2)".
    Failing that, the end of the parent paragraph -- appending to the end of the
    whole section instead puts the text past every later top-level paragraph,
    which costs two wrong operations rather than one.
    """
    spans = paragraph_spans(tokens)
    parts = addr.split(".")
    sibling_prefix = ".".join(parts[:-1])

    best = None
    for key, (s, e) in spans.items():
        kp = key.split(".")
        if ".".join(kp[:-1]) == sibling_prefix and len(kp) == len(parts) and key != addr:
            if best is None or e > best:
                best = e
    if best is None and sibling_prefix in spans:
        best = spans[sibling_prefix][1]
    at = best if best is not None else len(tokens)
    return tokens[:at] + chunk.split() + tokens[at:]


# ---------------------------------------------------------------------- baselines

def baseline_copy(before_text, blocks):
    return before_text


def baseline_printed(before_text, blocks):
    out = " ".join((b.get("printed") or "") for b in blocks).strip()
    return out or before_text


def baseline_append(before_text, blocks):
    return (before_text + " " + " ".join(
        (b.get("printed") or "") for b in blocks)).strip()


def baseline_splice(before_text, blocks):
    """Replace every addressed paragraph with the printed block wholesale --
    no '* * *' handling, no removals, no redesignation."""
    tokens = before_text.split()
    for blk in blocks:
        printed = split_printed(blk.get("printed") or "")
        for instr in blk["instructions"]:
            for addr in parse_addresses(instr) or [k for k in printed if k]:
                chunk = printed.get(addr)
                if chunk is None:
                    continue
                spans = paragraph_spans(tokens)
                if addr in spans:
                    s, e = spans[addr]
                    tokens = tokens[:s] + chunk.split() + tokens[e:]
    return " ".join(tokens)


ANY_DESIG = re.compile(r"^\((?:[0-9A-Za-z]{1,4})\)")


def baseline_regex_splice(before_text, blocks):
    """The first thing anyone writes: find the designator with a regular
    expression, replace up to the next designator of any kind.  No hierarchy, so
    it stops at nested sub-paragraphs and at cross-references alike."""
    toks = before_text.split()
    for blk in blocks:
        printed = (blk.get("printed") or "").strip()
        if not printed:
            continue
        for instr in blk["instructions"]:
            m = ADDR.search(instr)
            if not m:
                continue
            target = re.findall(r"(?:\([0-9A-Za-z]{1,4}\))+", m.group(1))[0]
            for i, t in enumerate(toks):
                if t.startswith(target):
                    end = next((j for j in range(i + 1, len(toks))
                                if ANY_DESIG.match(toks[j])), len(toks))
                    toks = toks[:i] + printed.split() + toks[end:]
                    break
    return " ".join(toks)

In [ ]:
print(f"stage 2, held out: {evaluate(val_ids, baseline_splice):.4f}")

## Stage 3 — partial replacements, deletions, redesignations

Three things stage 2 still gets wrong.

`* * *` in a printed block means *the rest of this paragraph is unchanged*. Splicing
the block in wholesale silently deletes the remainder, so we have to work out how
much of the original the printed fragment displaces — a sentence-boundary problem
in prose full of `U.S.C.`, `Pub. L.` and `Jan.`, where splitting on full stops does
not work.

Deletions print nothing at all, so there is no block to splice; the instruction
carries the text to remove in quotes.

And a run of instructions has to be applied in order, because redesignating (c)
through (f) as (d) through (g) moves the target of everything after it.

In [ ]:
print(f"stage 3, held out: {evaluate(val_ids, apply_instructions):.4f}")

for name, fn in [("do nothing", lambda b, bl: b),
                 ("printed block only",
                  lambda b, bl: " ".join((x.get('printed') or '') for x in bl) or b),
                 ("stage 1 naive splice", naive_apply),
                 ("stage 2 hierarchy", baseline_splice),
                 ("stage 3 full", apply_instructions)]:
    print(f"  {name:22s} {evaluate(val_ids, fn):.4f}")

The held-out numbers above are measured on items stage 3 was never inspected
against. Where it still loses is mostly tables: canonicalisation flattens them into
long token runs and addresses like *"revising Table 1 to paragraph (b)"* resolve
poorly against that.

In [ ]:
by_kind = {}
for i in val_ids:
    ins = " ".join(x for bl in items[i]["blocks"] for x in bl["instructions"]).lower()
    kind = ("table" if "table" in ins else
            "removal" if "remov" in ins else
            "redesignation" if "redesignat" in ins else
            "partial (* * *)" if "* * *" in (items[i]["blocks"][0].get("printed") or "") else
            "paragraph revision")
    by_kind.setdefault(kind, []).append(i)

for kind, ids_ in sorted(by_kind.items(), key=lambda kv: -len(kv[1])):
    print(f"  {kind:22s} n={len(ids_):4d}  {evaluate(ids_, apply_instructions):.4f}")

## Submission

In [ ]:
rows = []
for i in test["item_id"]:
    it = items[i]
    try:
        pred = apply_instructions(it["before_text"], it["blocks"]) or it["before_text"]
    except Exception:
        pred = it["before_text"]
    rows.append({"item_id": i, "after_text": normalise(pred)})

sub = pd.DataFrame(rows)
out = Path("working"); out.mkdir(exist_ok=True)
sub.to_csv(out / "submission.csv", index=False)

assert len(sub) == len(test)
assert sub["item_id"].is_unique
assert (sub["after_text"].str.strip() != "").all()
changed = sum(1 for r in rows if normalise(items[r["item_id"]]["before_text"]) != r["after_text"])
print(f"wrote {len(sub)} rows; changed {changed} of them "
      f"({changed/len(sub):.0%}) -- the rest we could not resolve and left alone")
sub.head(3)